# 01 — Linear Regression from Scratch (NumPy): Closed-Form + Gradient Descent

**Overview:** derive and implement least-squares linear/multiple regression two ways: the
closed-form normal equation, and gradient descent — and verify they agree.

**Required libraries:** `numpy`, `matplotlib`.

Connects to: `06_ml_fundamentals` (parameters vs hyperparameters) and Day 1's
`05_ml_mathematics` (gradient descent, matrix algebra). Leads to:
`02_linear_regression_sklearn.ipynb`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **MSE** | Mean Squared Error | Average of squared prediction errors; the standard regression loss. |
| **MAE** | Mean Absolute Error | Average of absolute errors; less outlier-sensitive than MSE. |
| **RMSE** | Root Mean Squared Error | Square root of MSE; in the same units as the target. |
| **R²** | Coefficient of determination | Fraction of target variance explained by the model (1 = perfect). |

## 0. Environment

In [1]:
import numpy as np
import matplotlib.pyplot as plt
print("numpy:", np.__version__)
rng = np.random.default_rng(42)


numpy: 2.5.3


## 1. Generate synthetic multiple-regression data

**Ground truth:** `y = 3*x1 - 2*x2 + 5 + noise` — we'll try to recover `w=[3,-2]`, `b=5`.

In [2]:
n = 200
X = rng.uniform(-5, 5, size=(n, 2))
true_w = np.array([3.0, -2.0])
true_b = 5.0
y = X @ true_w + true_b + rng.normal(scale=1.0, size=n)

print("X shape:", X.shape, "y shape:", y.shape)
print("true_w:", true_w, "true_b:", true_b)


X shape: (200, 2) y shape: (200,)
true_w: [ 3. -2.] true_b: 5.0


## 2. Closed-form least squares (the normal equation)

**Equation:** `w = (XᵗX)⁻¹ Xᵗy`, with a column of 1s appended to `X` so the bias is folded into `w`.

💡 **Intuition:** this is the exact point where the squared-error loss's gradient is zero —
solved directly via linear algebra, no iteration needed.

In [3]:
X_with_bias = np.column_stack([X, np.ones(n)])   # append a column of 1s for the bias term

w_closed_form = np.linalg.inv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
w_cf, b_cf = w_closed_form[:-1], w_closed_form[-1]

print("closed-form w:", w_cf.round(3), "(true:", true_w, ")")
print("closed-form b:", round(b_cf, 3), "(true:", true_b, ")")

assert np.allclose(w_cf, true_w, atol=0.3)
assert abs(b_cf - true_b) < 0.3


closed-form w: [ 3.037 -2.051] (true: [ 3. -2.] )
closed-form b: 4.959 (true: 5.0 )


## 3. Gradient descent (the general-purpose alternative)

**Equations:** `MSE = mean((Xw + b - y)^2)`, `∂MSE/∂w = (2/n) Xᵗ(Xw+b-y)`,
`∂MSE/∂b = (2/n) Σ(Xw+b-y)`.

In [4]:
w_gd = np.zeros(2)
b_gd = 0.0
lr = 0.01
mse_history = []

for epoch in range(2000):
    y_pred = X @ w_gd + b_gd
    error = y_pred - y
    mse = np.mean(error ** 2)
    mse_history.append(mse)

    grad_w = (2 / n) * (X.T @ error)
    grad_b = (2 / n) * error.sum()

    w_gd -= lr * grad_w
    b_gd -= lr * grad_b

print("gradient-descent w:", w_gd.round(3), "(true:", true_w, ")")
print("gradient-descent b:", round(b_gd, 3), "(true:", true_b, ")")
print("final MSE:", mse_history[-1])

assert np.allclose(w_gd, true_w, atol=0.3)
assert abs(b_gd - true_b) < 0.3

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(mse_history)
ax.set_xlabel("epoch"); ax.set_ylabel("MSE"); ax.set_title("Gradient descent convergence")
plt.show()


gradient-descent w: [ 3.037 -2.051] (true: [ 3. -2.] )
gradient-descent b: 4.959 (true: 5.0 )
final MSE: 0.958870674883402


## 4. Closed-form and gradient descent should agree

🧠 **Remember:** both methods minimize the exact same loss surface — they should land on
(approximately) the same weights, up to gradient descent's convergence tolerance.

In [5]:
print("closed-form w, b:", w_cf.round(4), b_cf.round(4))
print("gradient descent w, b:", w_gd.round(4), b_gd.round(4))

assert np.allclose(w_cf, w_gd, atol=0.1)
assert abs(b_cf - b_gd) < 0.2
print("\nBoth methods converge to (approximately) the same solution, as expected for a convex loss.")


closed-form w, b: [ 3.0375 -2.0508] 4.9592
gradient descent w, b: [ 3.0375 -2.0508] 4.9592

Both methods converge to (approximately) the same solution, as expected for a convex loss.


## 5. Evaluation metrics from scratch

**Equations:** `MSE = mean((ŷ-y)^2)`, `MAE = mean(|ŷ-y|)`, `RMSE = sqrt(MSE)`,
`R² = 1 - SS_res/SS_tot`.

In [6]:
y_pred_final = X @ w_cf + b_cf

mse = np.mean((y_pred_final - y) ** 2)
mae = np.mean(np.abs(y_pred_final - y))
rmse = np.sqrt(mse)
ss_res = np.sum((y - y_pred_final) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"MSE:  {mse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}  (same units as y, easier to interpret than MSE)")
print(f"R²:   {r2:.4f}  (close to 1.0 means the model explains most of the variance)")

# A model that always predicts the mean should score R² close to 0.
baseline_pred = np.full_like(y, y.mean())
baseline_r2 = 1 - np.sum((y - baseline_pred) ** 2) / ss_tot
print(f"\nbaseline (always predict mean) R²: {baseline_r2:.4f}  (should be ~0.0)")

assert r2 > 0.8
assert abs(baseline_r2) < 1e-9


MSE:  0.9589
MAE:  0.7714
RMSE: 0.9792  (same units as y, easier to interpret than MSE)
R²:   0.9909  (close to 1.0 means the model explains most of the variance)

baseline (always predict mean) R²: 0.0000  (should be ~0.0)


## 6. Connection to what's next

`02_linear_regression_sklearn.ipynb` fits the exact same data with `sklearn.linear_model.LinearRegression`
and confirms it matches this closed-form solution, then uses `sklearn.metrics` for the same
evaluation.

## 7. Interview questions
1. Derive (conceptually) why setting the gradient of MSE to zero gives the normal equation.
2. Why might `(XᵗX)⁻¹` fail to exist, and what causes that in practice?
3. Why does gradient descent approach the same answer as the closed form, for this convex loss?
4. What does R² < 0 mean about a model's quality?
5. When would MAE be a more appropriate reported metric than RMSE?

## 8. Exercises
1. Add a third, irrelevant feature (pure noise) and verify its learned weight is close to 0.
2. Make two features nearly collinear and observe what happens to the closed-form solution's stability.
3. Repeat gradient descent with a learning rate 100x higher and describe what happens to `mse_history`.

## 9. Summary

✅ Closed-form least squares and gradient descent solve the same convex optimization problem and
should agree; MSE/MAE/RMSE/R² each highlight a different aspect of error. Next:
`02_linear_regression_sklearn.ipynb`.